# 01 - Base de conhecimento e casos de teste
Baixa as leis federais (planalto.gov.br), quebra em trechos por artigo (com id, fonte, URL e licença) e valida os 50 casos de teste contra a base.
**Requisitos 01 (casos com 'não consta') e 02 (base em trechos com fonte).**

In [ ]:
# Setup (Colab: Ambiente de execução > Alterar tipo > GPU T4). Local: só rode a partir da raiz do repositório.
import os, sys, subprocess
REPO = "https://github.com/melo-zz/calcada-livre-genai.git"   # <- troque pelo link do repositório público da equipe
if os.path.exists("/content"):                                  # estamos no Colab
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("pasta de trabalho:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass

In [ ]:
from src.kb import construir_base
trechos, resumo = construir_base("base")

## Tabela da base (fonte, formato, licença, nº de trechos)

In [ ]:
import pandas as pd
tabela = pd.read_csv("base/fontes.csv")
tabela.loc[len(tabela)] = ["TOTAL", "", "", "", tabela["n_trechos"].sum()]
display(tabela)
tabela.to_markdown("base/tabela_fontes.md", index=False)
assert (tabela["n_trechos"][:-1] > 0).all(), "Alguma fonte falhou ao baixar: veja a mensagem ATENÇÃO acima (o site pode ter mudado a URL)."

## Amostra de trechos

In [ ]:
import random
for t in random.Random(SEED).sample(trechos, 4):
    print(t["id"], "|", t["fonte"], "|", t["url"]); print(t["texto"][:300], "\n")

## Casos de teste com gabarito
O gabarito é por evidência: cada caso aponta uma frase que precisa existir na base; os trechos que a contêm são os 'relevantes'. Casos sem evidência são listados abaixo (corrija a frase em `avaliacao/casos.py` e rode de novo).

In [ ]:
from avaliacao.casos import preparar
casos, invalidos = preparar(trechos)
print(f"{len(casos)} casos válidos | {sum(c['consta'] for c in casos)} 'consta' | {sum(not c['consta'] for c in casos)} 'não consta'")
print(f"{len(invalidos)} casos SEM evidência na base (excluídos):")
for c in invalidos:
    print(" -", c["pergunta"], "| evidência procurada:", repr(c["evidencia"]))

Se sobrarem poucos casos 'consta' (menos de ~35), a base não está com o texto esperado: me mande a lista acima que eu ajusto as evidências.